# Line 16 (Specialty Auto) — Loss Ratio vs NTR, Cat ExcludedRecreation of ngraf's `spl_lr` for the v9.4.0 / v1.5.0 refit.Single line, single purpose. New vs prior: **catastrophe losses removed from the numerator.**

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport statsmodels.formula.api as smfpd.set_option("display.width", 200)LINE  = 16KEYS  = ["ACTMO", "ACTYR", "ALINE", "CLINE", "COMPNY", "GEOST", "NTR"]DATA  = "data"                      # <-- folder holding the extractsBASE  = "ssenp_20260729-162408"     # <-- extract base name# Current production fit for line 16: [slope, intercept, ntr0_value]PROD_16 = [-0.02294, 0.6347, 1.0057]

## 1. Load claims

In [ ]:
clm = pd.read_csv(f"{DATA}/{BASE}c.csv")print(clm.shape)print(clm.columns.tolist())clm.head()

## 2. Cat indicatorCATCD is a single letter (A–Z, skipping I and O) or blank. **Non-blank = catastrophe.**

In [ ]:
clm["is_cat"] = clm["CATCD"].fillna("").astype(str).str.strip().ne("")print("cat codes present:", sorted(clm.loc[clm.is_cat, "CATCD"].str.strip().unique()))print(f"cat rows: {clm.is_cat.sum():,} of {len(clm):,}")# Did CATCD split cells into multiple rows? If yes, claims MUST be# re-aggregated to cell grain before merging to premium.n_cells = clm[KEYS].drop_duplicates().shape[0]print(f"rows {len(clm):,} vs unique cells {n_cells:,} -> grain changed: {len(clm) > n_cells}")

## 3. Cat share of losses — shareable without premium

In [ ]:
clm["TOTLOSS"] = clm["CMEXP"] + clm["CMLOSS"]split = (clm.groupby(["NTR", "is_cat"])["TOTLOSS"].sum()            .unstack(fill_value=0)            .rename(columns={False: "loss_ex_cat", True: "loss_cat"}))split["loss_all"]  = split.sum(axis=1)split["cat_share"] = split["loss_cat"] / split["loss_all"]tot = clm.groupby("is_cat")["TOTLOSS"].sum()print(f"TOTAL cat share of line {LINE} losses: {tot.get(True,0)/tot.sum():.1%}")split.round(4)

In [ ]:
ax = split["cat_share"].plot(kind="bar")ax.set_title(f"Line {LINE}: catastrophe share of incurred loss by NTR")ax.set_xlabel("NTR"); ax.set_ylabel("cat share of loss")plt.tight_layout(); plt.show()

## 4. Load premiumNeeds the matching `p` extract (`EEXP`, `EPREM`). Everything below requires it —without a denominator there is no loss ratio.

In [ ]:
try:    prem = pd.read_csv(f"{DATA}/{BASE}p.csv")    HAVE_PREM = True    print(prem.shape, prem.columns.tolist())except FileNotFoundError:    HAVE_PREM = False    print(f"!! {BASE}p.csv not found -- premium extract still needed from Nick.")    print("!! Sections 5+ cannot run. Section 3 above is still valid.")

## 5. Filter cat → re-aggregate → mergeOrder matters. CATCD adds grain to the claims file, so collapse back to celllevel **before** merging. Filtering after the merge fans out premium rows.

In [ ]:
def claims_at_cell_grain(df, mode):    """mode: 'all' | 'ex_cat' | 'cat_only'"""    sub = {"all": df, "ex_cat": df[~df.is_cat], "cat_only": df[df.is_cat]}[mode]    return sub.groupby(KEYS, as_index=False)[["CMEXP", "CMLOSS"]].sum()def build(prem, clm, mode):    cells = prem.merge(claims_at_cell_grain(clm, mode), on=KEYS,                       how="outer", indicator=True, validate="one_to_one")    for col in ["EEXP", "EPREM", "CMEXP", "CMLOSS"]:        cells[col] = cells[col].fillna(0.0)    cells["TOTLOSS"]    = cells["CMEXP"] + cells["CMLOSS"]    cells["loss_ratio"] = np.where(cells.EPREM > 0, cells.TOTLOSS / cells.EPREM, 0.0)    cells["w"]          = cells["EPREM"]    return cellsif HAVE_PREM:    # additivity check -- must hold or the cat filter is wrong    a  = claims_at_cell_grain(clm, "all")["CMLOSS"].sum()    ex = claims_at_cell_grain(clm, "ex_cat")["CMLOSS"].sum()    ct = claims_at_cell_grain(clm, "cat_only")["CMLOSS"].sum()    assert np.isclose(a, ex + ct, atol=0.01), "all != ex_cat + cat_only"    print(f"additivity OK  |  cat = {ct/a:.1%} of CMLOSS")    cells_all = build(prem, clm, "all")    cells_ex  = build(prem, clm, "ex_cat")    print(cells_ex["_merge"].value_counts().to_dict())

## 6. Reconciliation — what changed in losses

In [ ]:
if HAVE_PREM:    def by_ntr(cells, tag):        g = cells.groupby("NTR").agg(premium=("EPREM","sum"), loss=("TOTLOSS","sum"))        g[f"lr_{tag}"] = g["loss"] / g["premium"]        return g.rename(columns={"loss": f"loss_{tag}"})[[f"loss_{tag}", f"lr_{tag}"]]    recon = (cells_all.groupby("NTR")[["EPREM"]].sum().rename(columns={"EPREM":"premium"})             .join(by_ntr(cells_all, "all"))             .join(by_ntr(cells_ex,  "ex_cat")))    recon["lr_delta"] = recon["lr_ex_cat"] - recon["lr_all"]    display(recon.round(4))    print("overall LR  all: {:.4f}   ex-cat: {:.4f}".format(        recon.loss_all.sum()/recon.premium.sum(),        recon.loss_ex_cat.sum()/recon.premium.sum()))

## 7. FitWeighted least squares, `loss_ratio ~ NTR`, weights = earned premium.Line 16 is fit on **NTR > 0 only**; NTR = 0 gets a standalone weighted average,because new business drags the slope badly. That piecewise-vs-single-line choiceis the judgment call to confirm with Yinan.

In [ ]:
def fit(cells, tag):    d  = cells[(cells.ALINE == LINE) & (cells.EPREM > 0)]    dg = d[d.NTR > 0]                       # NTR>0 fit for line 16    lm = smf.wls("loss_ratio ~ NTR", data=dg, weights=dg["w"]).fit()    p, raw = float(lm.pvalues["NTR"]), float(lm.params["NTR"])    flat   = (p > 0.1) or (raw > 0)         # production fallback rule    slope  = 0.0 if flat else raw    icept  = (np.average(dg.loss_ratio, weights=dg.w) if flat              else float(lm.params["Intercept"]))    n0   = d[d.NTR == 0]    ntr0 = np.average(n0.loss_ratio, weights=n0.w) if len(n0) else np.nan    pts = (d.groupby("NTR")             .apply(lambda g: np.average(g.loss_ratio, weights=g.w),                    include_groups=False)             .rename("lr_wavg").reset_index())    print(f"[{tag}] p={p:.3g}  slope={slope:.5f}  intercept={icept:.5f}  "          f"NTR0={ntr0:.5f}  flat_fallback={flat}")    return dict(tag=tag, slope=slope, intercept=icept, ntr0=ntr0, pts=pts)if HAVE_PREM:    f_all = fit(cells_all, "with cat")    f_ex  = fit(cells_ex,  "ex cat")

## 8. Plot: production vs with-cat vs ex-cat

In [ ]:
if HAVE_PREM:    ntr  = np.arange(0, 10)    line = lambda f: np.where(ntr == 0, f["ntr0"], f["intercept"] + f["slope"]*ntr)    prod = np.where(ntr == 0, PROD_16[2], PROD_16[1] + PROD_16[0]*ntr)    plt.scatter(f_ex["pts"].NTR, f_ex["pts"].lr_wavg, label="observed ex-cat", zorder=3)    plt.plot(ntr, prod,        ":", color="red", label="production (v7)")    plt.plot(ntr, line(f_all), "--",             label="refit, with cat")    plt.plot(ntr, line(f_ex),  "-",              label="refit, ex cat")    plt.title(f"Line {LINE} Specialty Auto — loss ratio vs NTR")    plt.xlabel("NTR"); plt.ylabel("average loss ratio"); plt.legend()    plt.tight_layout(); plt.show()

## 9. Output for the pipeline

In [ ]:
if HAVE_PREM:    print(f"spl_fits[{LINE}] = [{f_ex['slope']:.6f}, {f_ex['intercept']:.6f}, {f_ex['ntr0']:.6f}]")    print(f'other_lr_tuple(line="{LINE}", slope_yr={f_ex["slope"]:.10f}, '          f'intercept={f_ex["intercept"]:.10f}),')

## Open items for Yinan1. **Premium extract missing** — only the `c` (claims) file is pulled. Need `...p.csv` with `EEXP`/`EPREM`.2. **Confirm every non-blank CATCD is a true catastrophe** (not non-cat weather / administrative).3. **ACTYR = 125** — is this years-since-1900 (2025)? Rebalancing window is 2024→present; extract currently looks like one year.4. **Fallback rule** — production tests `p > 0.1 or slope > 0`; the "Finalized Fits" cell tests only `p > 0.1`. Which is intended?5. **Where does the cat load get added back** downstream, since premium stays whole?6. **Line 78 Florida carve-out** — was cat-motivated. Re-test once cat is removed, or it double-counts.